# Notebook 05: Alpha Blending and Volume Rendering

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase1/05_alpha_blending.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand the volume rendering equation
2. Master front-to-back alpha blending
3. Implement transmittance computation
4. Learn how depth ordering affects rendering
5. Compare different blending strategies

**Estimated Time**: 60 minutes

**Prerequisites**: Notebook 04 (Differentiable Rendering)

---

## Setup

In [ ]:
import os
import sys

# Colab setup
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('3DGS-from-scratch'):
        !git clone https://github.com/ChunLI-666/3DGS-from-scratch.git
    os.chdir('3DGS-from-scratch')
    !pip install -q plotly ipywidgets

# Path setup
for path in ['../../src', '../src', './src']:
    full_path = os.path.abspath(path)
    if os.path.exists(os.path.join(full_path, 'gaussian')):
        sys.path.insert(0, full_path)
        break

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("Setup complete!")

## 1. The Volume Rendering Equation

### From NeRF to 3DGS

NeRF's volume rendering equation integrates along a ray:

$$C(\mathbf{r}) = \int_{t_n}^{t_f} T(t) \cdot \sigma(\mathbf{r}(t)) \cdot \mathbf{c}(\mathbf{r}(t), \mathbf{d}) \, dt$$

where:
- $T(t) = \exp\left(-\int_{t_n}^{t} \sigma(s) ds\right)$ is the **transmittance**
- $\sigma$ is the **density**
- $\mathbf{c}$ is the **color**

### Discretization

In practice, we discretize into $N$ samples:

$$C = \sum_{i=1}^{N} T_i \cdot \alpha_i \cdot c_i$$

where:
- $\alpha_i = 1 - \exp(-\sigma_i \delta_i)$ is the alpha (opacity) of sample $i$
- $T_i = \prod_{j=1}^{i-1} (1 - \alpha_j)$ is the transmittance up to sample $i$

In [ ]:
# Visualize the volume rendering equation
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Ray through volume
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(-1, 1)

# Draw ray
ax.arrow(0, 0, 9, 0, head_width=0.1, head_length=0.3, fc='black', ec='black')
ax.text(9.5, 0, 'ray', va='center')

# Draw samples
sample_positions = [2, 4, 5.5, 7, 8.5]
alphas = [0.2, 0.6, 0.4, 0.3, 0.1]
colors = ['red', 'green', 'blue', 'yellow', 'purple']

for pos, alpha, color in zip(sample_positions, alphas, colors):
    ax.axvline(pos, color=color, linewidth=5, alpha=alpha)
    ax.text(pos, -0.5, f'α={alpha}', ha='center', fontsize=9)

ax.text(0.5, 0.8, 'Camera', fontsize=10, ha='center')
ax.scatter(0.5, 0, s=100, c='black', marker='s')
ax.set_title('Ray Through Colored Samples')
ax.axis('off')

# 2. Transmittance decay
ax = axes[1]
t = np.linspace(0, 10, 100)
sigma = 0.3  # Density
T = np.exp(-sigma * t)
ax.plot(t, T, 'b-', linewidth=2)
ax.fill_between(t, T, alpha=0.3)
ax.set_xlabel('Distance along ray')
ax.set_ylabel('Transmittance T(t)')
ax.set_title('Transmittance Decay\n$T(t) = e^{-\sigma \cdot t}$')
ax.set_ylim(0, 1.1)
ax.grid(True, alpha=0.3)

# 3. Alpha blending formula
ax = axes[2]
ax.text(0.5, 0.9, 'Volume Rendering Equation', ha='center', fontsize=14, fontweight='bold', transform=ax.transAxes)
ax.text(0.5, 0.7, r'$C = \sum_{i=1}^{N} T_i \cdot \alpha_i \cdot c_i$', ha='center', fontsize=16, transform=ax.transAxes)
ax.text(0.5, 0.5, 'where', ha='center', fontsize=12, transform=ax.transAxes)
ax.text(0.5, 0.35, r'$T_i = \prod_{j=1}^{i-1} (1 - \alpha_j)$', ha='center', fontsize=14, transform=ax.transAxes)
ax.text(0.5, 0.15, '(Transmittance = product of\n"how much light passes through")', ha='center', fontsize=10, transform=ax.transAxes)
ax.axis('off')

plt.tight_layout()
plt.show()

## 2. Understanding Transmittance

### What is Transmittance?

Transmittance $T_i$ represents **how much light can reach sample $i$** after passing through all previous samples.

$$T_i = \prod_{j=1}^{i-1} (1 - \alpha_j)$$

### Properties:

1. $T_1 = 1$ (first sample receives all light)
2. $T_i$ decreases monotonically as $i$ increases
3. If any $\alpha_j = 1$, then $T_i = 0$ for all $i > j$

In [ ]:
def compute_transmittance(alphas):
    """
    Compute transmittance for each sample.
    
    Args:
        alphas: [N] alpha values, sorted front-to-back
    
    Returns:
        [N] transmittance values
    """
    N = len(alphas)
    transmittance = torch.ones(N)
    
    # T_i = prod_{j<i}(1 - alpha_j)
    cumulative = 1.0
    for i in range(N):
        transmittance[i] = cumulative
        cumulative = cumulative * (1 - alphas[i])
    
    return transmittance


# Example: Different alpha distributions
print("Transmittance Examples:")
print("=" * 60)

alpha_examples = {
    'Uniform low (0.1)': torch.tensor([0.1, 0.1, 0.1, 0.1, 0.1]),
    'Uniform high (0.5)': torch.tensor([0.5, 0.5, 0.5, 0.5, 0.5]),
    'Opaque first': torch.tensor([0.9, 0.5, 0.3, 0.2, 0.1]),
    'Opaque last': torch.tensor([0.1, 0.2, 0.3, 0.5, 0.9]),
    'One opaque': torch.tensor([0.1, 0.1, 1.0, 0.1, 0.1]),
}

fig, axes = plt.subplots(1, len(alpha_examples), figsize=(15, 3))

for ax, (name, alphas) in zip(axes, alpha_examples.items()):
    T = compute_transmittance(alphas)
    weights = T * alphas
    
    x = np.arange(len(alphas))
    width = 0.35
    
    ax.bar(x - width/2, alphas.numpy(), width, label='α', color='blue', alpha=0.7)
    ax.bar(x + width/2, T.numpy(), width, label='T', color='orange', alpha=0.7)
    ax.plot(x, weights.numpy(), 'g-o', linewidth=2, markersize=8, label='T·α')
    
    ax.set_xlabel('Sample index')
    ax.set_ylim(0, 1.1)
    ax.set_title(name)
    ax.legend(fontsize=8)
    ax.set_xticks(x)

plt.tight_layout()
plt.show()

print("\nObservations:")
print("- Transmittance always starts at 1")
print("- Higher alphas cause faster transmittance decay")
print("- When alpha=1, transmittance drops to 0 for all subsequent samples")
print("- Weight (T·α) determines contribution to final color")

## 3. Front-to-Back Alpha Blending

### The Algorithm

```python
C = 0  # Accumulated color
T = 1  # Remaining transmittance

for each sample i (front to back):
    C = C + T * alpha[i] * color[i]
    T = T * (1 - alpha[i])

C = C + T * background  # Add background
```

### Why Front-to-Back?

1. **Early termination**: Can stop when $T < \epsilon$
2. **Natural occlusion**: Front objects block back objects
3. **Correct depth ordering**: Critical for correct results

In [ ]:
def alpha_blend_front_to_back(colors, alphas, background=None):
    """
    Perform front-to-back alpha blending.
    
    Args:
        colors: [N, 3] RGB colors (sorted front-to-back)
        alphas: [N] alpha values (sorted front-to-back)
        background: [3] background color
    
    Returns:
        [3] blended color
    """
    if background is None:
        background = torch.ones(3)
    
    C = torch.zeros(3)
    T = 1.0
    
    for i in range(len(alphas)):
        weight = T * alphas[i]
        C = C + weight * colors[i]
        T = T * (1 - alphas[i])
    
    # Add background
    C = C + T * background
    
    return C


# Example with colored layers
colors = torch.tensor([
    [1.0, 0.0, 0.0],  # Red (front)
    [0.0, 1.0, 0.0],  # Green
    [0.0, 0.0, 1.0],  # Blue (back)
])

# Different alpha configurations
alpha_configs = [
    ('All transparent (0.3)', torch.tensor([0.3, 0.3, 0.3])),
    ('All semi-opaque (0.7)', torch.tensor([0.7, 0.7, 0.7])),
    ('Front opaque', torch.tensor([1.0, 0.5, 0.5])),
    ('Back opaque', torch.tensor([0.3, 0.3, 1.0])),
    ('All opaque', torch.tensor([1.0, 1.0, 1.0])),
]

fig, axes = plt.subplots(2, len(alpha_configs), figsize=(15, 6))

for col, (name, alphas) in enumerate(alpha_configs):
    # Show layers
    ax_top = axes[0, col]
    for i, (c, a) in enumerate(zip(colors, alphas)):
        rect = Rectangle((0.2, 0.1 + i*0.25), 0.6, 0.2, 
                         facecolor=c.numpy(), alpha=a.item(),
                         edgecolor='black', linewidth=2)
        ax_top.add_patch(rect)
        ax_top.text(0.9, 0.2 + i*0.25, f'α={a.item():.1f}', fontsize=10)
    ax_top.set_xlim(0, 1.2)
    ax_top.set_ylim(0, 1)
    ax_top.set_title(name)
    ax_top.axis('off')
    ax_top.text(0.5, 0.05, '← Front', ha='center', fontsize=8)
    ax_top.text(0.5, 0.95, 'Back →', ha='center', fontsize=8)
    
    # Show result
    ax_bot = axes[1, col]
    result = alpha_blend_front_to_back(colors, alphas)
    rect = Rectangle((0.1, 0.1), 0.8, 0.8, 
                     facecolor=result.numpy().clip(0, 1),
                     edgecolor='black', linewidth=2)
    ax_bot.add_patch(rect)
    ax_bot.set_xlim(0, 1)
    ax_bot.set_ylim(0, 1)
    ax_bot.set_title(f'Result: RGB={result.numpy().round(2)}')
    ax_bot.axis('off')

plt.suptitle('Front-to-Back Alpha Blending', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4. Depth Ordering Matters!

The order of samples is **critical** for correct alpha blending.

In [ ]:
# Demonstrate importance of correct depth ordering

# Three overlapping semi-transparent circles
colors = torch.tensor([
    [1.0, 0.0, 0.0],  # Red
    [0.0, 1.0, 0.0],  # Green
    [0.0, 0.0, 1.0],  # Blue
])

# All with same alpha
alphas = torch.tensor([0.7, 0.7, 0.7])

# Different orderings
orderings = [
    ('R-G-B', [0, 1, 2]),
    ('B-G-R', [2, 1, 0]),
    ('G-R-B', [1, 0, 2]),
    ('B-R-G', [2, 0, 1]),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, (name, order) in zip(axes, orderings):
    ordered_colors = colors[order]
    ordered_alphas = alphas[order]
    
    result = alpha_blend_front_to_back(ordered_colors, ordered_alphas)
    
    # Draw the layers
    for i, (c, a, idx) in enumerate(zip(ordered_colors, ordered_alphas, order)):
        label = ['R', 'G', 'B'][idx]
        offset = i * 0.15
        rect = Rectangle((0.15 + offset, 0.15 + offset), 0.4, 0.4,
                         facecolor=c.numpy(), alpha=a.item(),
                         edgecolor='black', linewidth=2)
        ax.add_patch(rect)
        ax.text(0.35 + offset, 0.35 + offset, label, fontsize=14, 
               ha='center', va='center', fontweight='bold')
    
    # Show result
    result_rect = Rectangle((0.6, 0.2), 0.3, 0.3,
                            facecolor=result.numpy().clip(0, 1),
                            edgecolor='black', linewidth=2)
    ax.add_patch(result_rect)
    ax.text(0.75, 0.1, '= Result', ha='center', fontsize=10)
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 0.8)
    ax.set_title(f'Order: {name}\nRGB={result.numpy().round(2)}')
    ax.axis('off')
    ax.text(0.35, 0.02, 'Front→Back', fontsize=8, ha='center')

plt.suptitle('Different Depth Orderings Produce Different Results!', 
            fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Key insight: Same colors and alphas, different orderings = different results!")
print("This is why 3DGS must sort Gaussians by depth before rendering.")

## 5. Full Alpha Blending Implementation

Let's implement a complete, vectorized alpha blending function.

In [ ]:
def alpha_blend_vectorized(colors, alphas, background=None):
    """
    Vectorized front-to-back alpha blending.
    
    Args:
        colors: [N, C] colors (front-to-back)
        alphas: [N] alphas (front-to-back)
        background: [C] background color
    
    Returns:
        [C] blended color
    """
    N = alphas.shape[0]
    C = colors.shape[1]
    
    if background is None:
        background = torch.ones(C, dtype=colors.dtype, device=colors.device)
    
    # Compute transmittance: T_i = prod_{j<i}(1 - alpha_j)
    one_minus_alpha = 1 - alphas
    # Exclusive cumulative product (shift by 1)
    transmittance = torch.ones(N, dtype=alphas.dtype, device=alphas.device)
    transmittance[1:] = torch.cumprod(one_minus_alpha[:-1], dim=0)
    
    # Weights: w_i = T_i * alpha_i
    weights = transmittance * alphas  # [N]
    
    # Weighted sum of colors
    blended = (weights.unsqueeze(1) * colors).sum(dim=0)  # [C]
    
    # Final transmittance for background
    final_T = one_minus_alpha.prod()
    blended = blended + final_T * background
    
    return blended, weights, transmittance


# Test and visualize
N = 10
colors = torch.rand(N, 3)
alphas = torch.rand(N) * 0.3 + 0.2  # Random alphas between 0.2 and 0.5

result, weights, transmittance = alpha_blend_vectorized(colors, alphas)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Show individual contributions
ax = axes[0]
for i in range(N):
    ax.bar(i, weights[i].item(), color=colors[i].numpy(), edgecolor='black')
ax.set_xlabel('Sample index (front to back)')
ax.set_ylabel('Weight (T·α)')
ax.set_title('Individual Contributions')

# Show alpha and transmittance
ax = axes[1]
x = np.arange(N)
ax.plot(x, alphas.numpy(), 'b-o', label='α (alpha)')
ax.plot(x, transmittance.numpy(), 'r-s', label='T (transmittance)')
ax.plot(x, weights.numpy(), 'g-^', label='T·α (weight)')
ax.set_xlabel('Sample index')
ax.set_ylabel('Value')
ax.set_title('Alpha, Transmittance, and Weights')
ax.legend()
ax.grid(True, alpha=0.3)

# Show cumulative color
ax = axes[2]
cumulative = torch.zeros(N+1, 3)
for i in range(N):
    cumulative[i+1] = cumulative[i] + weights[i] * colors[i]
# Add background
final_T = (1 - alphas).prod()
cumulative[-1] = cumulative[-1] + final_T * torch.ones(3)

for c, label in zip(range(3), ['Red', 'Green', 'Blue']):
    ax.plot(np.arange(N+1), cumulative[:, c].numpy(), '-o', label=label)

ax.set_xlabel('Samples processed')
ax.set_ylabel('Accumulated color')
ax.set_title('Cumulative Color Accumulation')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal blended color: RGB = {result.numpy().round(3)}")
print(f"Sum of weights: {weights.sum().item():.3f} (+ {final_T.item():.3f} for background = {weights.sum().item() + final_T.item():.3f})")

## 6. Alpha Blending for Images

Now let's extend to full image rendering with multiple Gaussians per pixel.

In [ ]:
def render_gaussians_with_alpha_blending(
    means,         # [N, 2] - 2D centers
    covariances,   # [N, 2, 2] - 2D covariances
    opacities,     # [N] - base opacities
    colors,        # [N, 3] - RGB colors
    depths,        # [N] - depths for sorting
    height, width,
    background=None,
):
    """
    Render multiple Gaussians with proper alpha blending.
    """
    if background is None:
        background = torch.ones(3)
    
    N = means.shape[0]
    
    # Sort by depth (front to back)
    order = torch.argsort(depths)
    means = means[order]
    covariances = covariances[order]
    opacities = opacities[order]
    colors = colors[order]
    
    # Create coordinate grid
    y = torch.arange(height, dtype=torch.float32)
    x = torch.arange(width, dtype=torch.float32)
    y_grid, x_grid = torch.meshgrid(y, x, indexing='ij')
    
    # Initialize
    image = torch.zeros(height, width, 3)
    transmittance = torch.ones(height, width)
    
    for i in range(N):
        mean = means[i]
        cov = covariances[i]
        base_opacity = opacities[i]
        color = colors[i]
        
        # Compute Gaussian values
        cov_inv = torch.linalg.inv(cov)
        dx = x_grid - mean[0]
        dy = y_grid - mean[1]
        
        mahal = (cov_inv[0, 0] * dx * dx + 
                 (cov_inv[0, 1] + cov_inv[1, 0]) * dx * dy +
                 cov_inv[1, 1] * dy * dy)
        
        gaussian_value = torch.exp(-0.5 * mahal)
        
        # Alpha = gaussian_value * base_opacity
        alpha = gaussian_value * base_opacity
        
        # Contribution: weight = T * alpha
        weight = transmittance * alpha
        
        # Add weighted color
        for c in range(3):
            image[:, :, c] = image[:, :, c] + weight * color[c]
        
        # Update transmittance
        transmittance = transmittance * (1 - alpha)
    
    # Add background
    for c in range(3):
        image[:, :, c] = image[:, :, c] + transmittance * background[c]
    
    return torch.clamp(image, 0, 1), transmittance


# Create test scene with overlapping Gaussians
means = torch.tensor([
    [40., 40.],
    [60., 50.],
    [50., 60.],
])

covariances = torch.stack([
    torch.tensor([[300., 0.], [0., 200.]]),
    torch.tensor([[200., 50.], [50., 300.]]),
    torch.tensor([[250., -30.], [-30., 250.]]),
])

opacities = torch.tensor([0.8, 0.7, 0.75])

colors = torch.tensor([
    [1., 0., 0.],  # Red
    [0., 1., 0.],  # Green
    [0., 0., 1.],  # Blue
])

# Test with different depth orderings
depth_configs = [
    ('R front, B back', torch.tensor([1., 2., 3.])),
    ('B front, R back', torch.tensor([3., 2., 1.])),
    ('G front, others same', torch.tensor([2., 1., 2.])),
]

fig, axes = plt.subplots(1, len(depth_configs), figsize=(15, 4))

for ax, (name, depths) in zip(axes, depth_configs):
    image, final_T = render_gaussians_with_alpha_blending(
        means, covariances, opacities, colors, depths,
        height=100, width=100
    )
    ax.imshow(image.numpy())
    ax.set_title(f'{name}\nDepths: {depths.tolist()}')
    ax.axis('off')

plt.suptitle('Same Scene, Different Depth Orderings', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Early Termination for Efficiency

When transmittance becomes very small, we can stop processing more Gaussians.

In [ ]:
def render_with_early_termination(
    means, covariances, opacities, colors, depths,
    height, width,
    termination_threshold=0.01,
    background=None,
):
    """
    Render with early termination when transmittance is low.
    """
    if background is None:
        background = torch.ones(3)
    
    N = means.shape[0]
    
    # Sort by depth
    order = torch.argsort(depths)
    means = means[order]
    covariances = covariances[order]
    opacities = opacities[order]
    colors = colors[order]
    
    # Create coordinate grid
    y = torch.arange(height, dtype=torch.float32)
    x = torch.arange(width, dtype=torch.float32)
    y_grid, x_grid = torch.meshgrid(y, x, indexing='ij')
    
    # Initialize
    image = torch.zeros(height, width, 3)
    transmittance = torch.ones(height, width)
    
    # Track which pixels need more processing
    active_mask = torch.ones(height, width, dtype=torch.bool)
    gaussians_processed = torch.zeros(height, width, dtype=torch.int32)
    
    for i in range(N):
        # Check if any pixels still active
        if not active_mask.any():
            print(f"Early termination at Gaussian {i}/{N}")
            break
        
        mean = means[i]
        cov = covariances[i]
        base_opacity = opacities[i]
        color = colors[i]
        
        # Compute Gaussian values (only for active pixels for efficiency)
        cov_inv = torch.linalg.inv(cov)
        dx = x_grid - mean[0]
        dy = y_grid - mean[1]
        
        mahal = (cov_inv[0, 0] * dx * dx + 
                 (cov_inv[0, 1] + cov_inv[1, 0]) * dx * dy +
                 cov_inv[1, 1] * dy * dy)
        
        gaussian_value = torch.exp(-0.5 * mahal)
        alpha = gaussian_value * base_opacity
        
        # Only process active pixels
        weight = torch.where(active_mask, transmittance * alpha, torch.zeros_like(alpha))
        
        for c in range(3):
            image[:, :, c] = image[:, :, c] + weight * color[c]
        
        # Update transmittance
        transmittance = torch.where(active_mask, transmittance * (1 - alpha), transmittance)
        
        # Track processing
        gaussians_processed = torch.where(active_mask, gaussians_processed + 1, gaussians_processed)
        
        # Update active mask
        active_mask = transmittance > termination_threshold
    
    # Add background to remaining
    for c in range(3):
        image[:, :, c] = image[:, :, c] + transmittance * background[c]
    
    return torch.clamp(image, 0, 1), gaussians_processed


# Create many overlapping Gaussians
N = 50
torch.manual_seed(42)
means = torch.rand(N, 2) * 80 + 10  # Random positions in [10, 90]
covariances = torch.stack([torch.eye(2) * (100 + torch.rand(1).item() * 200) for _ in range(N)])
opacities = torch.rand(N) * 0.3 + 0.3  # Random opacities in [0.3, 0.6]
colors = torch.rand(N, 3)
depths = torch.rand(N) * 10

# Render with different thresholds
thresholds = [0.5, 0.1, 0.01, 0.001]

fig, axes = plt.subplots(2, len(thresholds), figsize=(16, 8))

for col, thresh in enumerate(thresholds):
    image, n_processed = render_with_early_termination(
        means, covariances, opacities, colors, depths,
        height=100, width=100,
        termination_threshold=thresh
    )
    
    axes[0, col].imshow(image.numpy())
    axes[0, col].set_title(f'Threshold = {thresh}')
    axes[0, col].axis('off')
    
    im = axes[1, col].imshow(n_processed.numpy(), cmap='viridis', vmin=0, vmax=N)
    axes[1, col].set_title(f'Gaussians processed per pixel\nAvg: {n_processed.float().mean():.1f}')
    axes[1, col].axis('off')
    plt.colorbar(im, ax=axes[1, col], fraction=0.046, pad=0.04)

plt.suptitle(f'Early Termination (Total: {N} Gaussians)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nKey insight: Higher threshold = fewer Gaussians processed = faster but less accurate")

## 8. Gradient Through Alpha Blending

For training, we need gradients through the entire blending operation.

In [ ]:
# Demonstrate gradient computation through alpha blending

# Simple case: 3 samples with learnable parameters
colors = torch.tensor([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
], requires_grad=True)

alphas = torch.tensor([0.5, 0.5, 0.5], requires_grad=True)

# Target: we want the result to be yellow [1, 1, 0]
target = torch.tensor([1.0, 1.0, 0.0])

# Forward pass with detailed tracking
N = alphas.shape[0]
transmittance = torch.ones(N)
transmittance[1:] = torch.cumprod(1 - alphas[:-1], dim=0)

weights = transmittance * alphas
result = (weights.unsqueeze(1) * colors).sum(dim=0)

# Add background contribution
final_T = (1 - alphas).prod()
background = torch.ones(3)
result = result + final_T * background

# Loss
loss = F.mse_loss(result, target)

# Backward
loss.backward()

print("Gradient Analysis for Alpha Blending:")
print("=" * 60)
print(f"\nTarget color: {target.tolist()}")
print(f"Result color: {result.detach().tolist()}")
print(f"Loss: {loss.item():.4f}")

print(f"\nAlphas: {alphas.detach().tolist()}")
print(f"Transmittance: {transmittance.detach().tolist()}")
print(f"Weights: {weights.detach().tolist()}")

print(f"\nGradients:")
print(f"  d(loss)/d(colors):\n{colors.grad}")
print(f"  d(loss)/d(alphas): {alphas.grad}")

print("\nInterpretation:")
print("- Color gradients show how to adjust each sample's color")
print("- Alpha gradients show whether to increase/decrease opacity")
print("- Red & green should become more opaque (more yellow)")
print("- Blue should become more transparent (less blue in result)")

In [ ]:
# Optimize alpha values to achieve a target color

colors_fixed = torch.tensor([
    [1.0, 0.0, 0.0],  # Red
    [0.0, 1.0, 0.0],  # Green
    [0.0, 0.0, 1.0],  # Blue
])

# Learnable alphas (in logit space for unconstrained optimization)
alphas_logit = torch.zeros(3, requires_grad=True)

# Target: pure green
target = torch.tensor([0.0, 0.8, 0.0])

optimizer = torch.optim.Adam([alphas_logit], lr=0.2)

history = []
for i in range(100):
    optimizer.zero_grad()
    
    # Convert logits to alphas
    alphas = torch.sigmoid(alphas_logit)
    
    # Alpha blending
    N = alphas.shape[0]
    transmittance = torch.ones(N)
    transmittance[1:] = torch.cumprod(1 - alphas[:-1], dim=0)
    
    weights = transmittance * alphas
    result = (weights.unsqueeze(1) * colors_fixed).sum(dim=0)
    
    final_T = (1 - alphas).prod()
    result = result + final_T * torch.ones(3)
    
    # Loss
    loss = F.mse_loss(result, target)
    
    loss.backward()
    optimizer.step()
    
    if i % 10 == 0:
        history.append({
            'iter': i,
            'alphas': alphas.detach().clone(),
            'result': result.detach().clone(),
            'loss': loss.item(),
        })

# Visualize
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Loss
axes[0].plot([h['loss'] for h in history], 'b-o')
axes[0].set_xlabel('Iteration (x10)')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

# Alphas evolution
axes[1].plot([h['alphas'][0].item() for h in history], 'r-o', label='Red α')
axes[1].plot([h['alphas'][1].item() for h in history], 'g-s', label='Green α')
axes[1].plot([h['alphas'][2].item() for h in history], 'b-^', label='Blue α')
axes[1].set_xlabel('Iteration (x10)')
axes[1].set_ylabel('Alpha')
axes[1].set_title('Alpha Values')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Target
rect = Rectangle((0.1, 0.1), 0.8, 0.8, facecolor=target.numpy(), edgecolor='black', linewidth=2)
axes[2].add_patch(rect)
axes[2].set_xlim(0, 1)
axes[2].set_ylim(0, 1)
axes[2].set_title(f'Target: {target.tolist()}')
axes[2].axis('off')

# Result
final_result = history[-1]['result']
rect = Rectangle((0.1, 0.1), 0.8, 0.8, facecolor=final_result.numpy().clip(0, 1), edgecolor='black', linewidth=2)
axes[3].add_patch(rect)
axes[3].set_xlim(0, 1)
axes[3].set_ylim(0, 1)
axes[3].set_title(f'Result: {final_result.numpy().round(2).tolist()}')
axes[3].axis('off')

plt.suptitle('Optimizing Alpha Values to Match Target Color', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nFinal alphas: R={history[-1]['alphas'][0]:.3f}, G={history[-1]['alphas'][1]:.3f}, B={history[-1]['alphas'][2]:.3f}")
print("The optimizer learned to make Green more opaque and Red/Blue more transparent!")

## 9. Summary: Alpha Blending in 3DGS

### Key Equations

| Equation | Formula | Description |
|----------|---------|-------------|
| Transmittance | $T_i = \prod_{j<i} (1 - \alpha_j)$ | Light remaining after $i-1$ samples |
| Weight | $w_i = T_i \cdot \alpha_i$ | Contribution of sample $i$ |
| Color | $C = \sum_i w_i \cdot c_i + T_N \cdot c_{bg}$ | Final blended color |

### Algorithm: Front-to-Back Blending

```python
# Initialize
C = 0  # Accumulated color
T = 1  # Transmittance

# Sort Gaussians by depth
gaussians = sort_by_depth(gaussians)

# Blend front to back
for g in gaussians:
    alpha = g.opacity * gaussian_value(pixel, g)
    C += T * alpha * g.color
    T *= (1 - alpha)
    
    # Optional early termination
    if T < threshold:
        break

# Add background
C += T * background
```

### Key Properties

1. **Order matters**: Different depth orderings produce different results
2. **Transmittance decay**: Each sample blocks some light from reaching further samples
3. **Differentiable**: Gradients flow through all operations
4. **Early termination**: Can skip samples when transmittance is low

---

## Key Takeaways

1. Alpha blending is the core rendering operation in 3DGS
2. Correct depth ordering is essential
3. Transmittance tracks remaining light visibility
4. Early termination improves efficiency
5. All operations are differentiable for optimization

---

## Next Steps

In the next notebook, we'll learn about **Spherical Harmonics** for view-dependent appearance:

**[06_spherical_harmonics.ipynb](./06_spherical_harmonics.ipynb)** - Spherical Harmonics for View-Dependent Color